# PMAPS Workshop: IDAES-GTEP, Session 1

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/idaes/idaes-gtep.git/main?urlpath=%2Fdoc%2Ftree%2Fdocs%2Fsource%2Ftutorials%2F5bus%2Ftutorial_5bus.ipynb)

Welcome! In this tutorial, we will demonstrate how to get started with IDAES Generation and Transmission Expansion Planning (GTEP) using the PJM 5-bus test case as an example. This tutorial will demonstrate the minimal setup to get started with GTEP. Then, it will showcase configuration options in GTEP and how they can be used to answer capacity expansion planning questions.

The following steps to run GTEP will be demonstrated in this notebook:
1. Reading in data (using the `ExpansionPlanningData` class)
2. Creating the model (using the `ExpansionPlanningModel` class)
3. Solving the model (using Pyomo utilities)
4. Exploring results (both manually and with the `ExpansionPlanningSolution` class)

Additional IDAES-GTEP resources:
- https://github.com/IDAES/idaes-gtep
- https://idaes-gtep.readthedocs.io/en/latest/index.html

Other tools leveraged by IDAES-GTEP in this tutorial:
- Prescient: https://github.com/grid-parity-exchange/prescient
- EGRET: https://github.com/grid-parity-exchange/egret
- Pyomo: https://github.com/Pyomo/pyomo
- HiGHS: https://ergo-code.github.io/HiGHS/stable/

Before anything else, we need to install a solver. More on this later--for now, just run the cell below, then restart the kernel and continue through the notebook.

In [ ]:
%pip install highspy

## 1. Reading in data

In [2]:
# suppressing some logs/warnings
import logging
import warnings
logging.getLogger().setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

### 1a. The `ExpansionPlanningData` class

The `ExpansionPlanningData` class is our starting point. It allows us to define the temporal parameters of our model and reads in data defining the grid assets and how they are connected.

Its constructor takes the following arguments, all of which are optional:

| Name | Type | Default value | Description |
| --- | --- | --- | --- |
| `stages` | `int` | `2` | Number of investment periods |
| `num_reps` | `int` | `4` | Number of representative periods in each investment period |
| `num_commit` | `int` | `24` | Number of commitment periods in each representative period |
| `num_dispatch` | `int` | `1` | Number of dispatch periods in each commitment period |
| `duration_representative_period` | `int` | `24` | Duration of each representative period (in hours) |
| `save_period_structure_file` | `bool` | `False` | Whether to save a period structure file |
| `period_structure_json_file` | `pathlib.Path` or `str` | `None` | Path to an existing period structure file |

Let's pick some values and create our `ExpansionPlanningData` object:

In [3]:
from gtep.gtep_data import ExpansionPlanningData

data_object = ExpansionPlanningData(
    stages=1,
    num_reps=4,
    num_commit=6,
    num_dispatch=4,
    duration_representative_period=6,
)

Interactive Python mode detected; using default matplotlib backend for plotting.


Once the `ExpansionPlanningData` object is instantiated, we must point it to a directory containing data that will define our model setup. GTEP expects the input data to have a particular form, including specific filenames. See the summary below:

| Filename | Required for GTEP? | Description |
| --- | --- | -- |
| `branch.csv` | Yes | Describes which buses each branch connects to; resistance; etc. |
| `bus.csv` | Yes | Bus ID along with bus properties (e.g., load) |
| `DAY_AHEAD_load.csv` | Yes | Load at each time step |
| `DAY_AHEAD_renewables.csv` | Yes | Renewable generation at each time step |
| `gen.csv` | Yes | Maps generators to buses; includes generator properties (e.g., technology type) |
| `initial_status.csv` | No | Describes the initial state of each generator |
| `reserves.csv` | No | Requirements on surplus generation capacity within a particular area |
| `simulation_objects.csv` | Yes | Several parameters required to read in data (start date, end date, etc.) |
| `timeseries_pointers.csv` | Yes | Mapping between generators/loads and respective day-ahead files |

For more information on most of these data files, see here: https://prescient.readthedocs.io/en/latest/reference/file_formats/rts-gmlc/index.html

The IDAES-GTEP GitHub repository contains several example datasets, including one for the PJM 5-bus test case. Below, we navigate to this directory and list the data files it contains:

In [4]:
from pathlib import Path
from IPython.display import Markdown, display

def display_markdown_of_paths(paths):
    display(Markdown("\n".join(
        f"- [{f.name}]({f.as_posix()})"
        for f in paths
    )))

data_path = Path("../../../../gtep/data/5bus")
display_markdown_of_paths(data_path.glob("*.csv"))

- [bus.csv](../../../../gtep/data/5bus/bus.csv)
- [gen.csv](../../../../gtep/data/5bus/gen.csv)
- [branch.csv](../../../../gtep/data/5bus/branch.csv)
- [REAL_TIME_renewables.csv](../../../../gtep/data/5bus/REAL_TIME_renewables.csv)
- [DAY_AHEAD_load.csv](../../../../gtep/data/5bus/DAY_AHEAD_load.csv)
- [initial_status.csv](../../../../gtep/data/5bus/initial_status.csv)
- [timeseries_pointers.csv](../../../../gtep/data/5bus/timeseries_pointers.csv)
- [storage.csv](../../../../gtep/data/5bus/storage.csv)
- [DAY_AHEAD_renewables.csv](../../../../gtep/data/5bus/DAY_AHEAD_renewables.csv)
- [REAL_TIME_load.csv](../../../../gtep/data/5bus/REAL_TIME_load.csv)
- [reserves.csv](../../../../gtep/data/5bus/reserves.csv)
- [simulation_objects.csv](../../../../gtep/data/5bus/simulation_objects.csv)

Try opening a few of these files to see their contents, for instance:
- `branch.csv`: You should see 7 different branches (AKA lines), each with properties with resistance (R) and reactance (X).
- `gen.csv`: You should see 12 different generators, each with an associated bus along with generator properties (unit type/technology, minimum power, maximum power, fuel cost, etc.)

Once we have the path to our data directory, we pass it into the `load_prescient` method of our `ExpansionPlanningData` object, which uses the data loader from the production cost modeling platform Prescient.

The following table summarizes the arguments for this method, which determine how the Prescient data loader is used:

| Name | Type | Default value | Description |
| --- | --- | --- | --- |
| `data_path` | `pathlib.Path` or `str` | - | Path to directory containing the data |
| `representative_dates` | `list[str]` | `None` (dates automatically chosen) | Representative dates to use; if provided, must have length `num_reps` |
| `representative_weights` | `list[float\|int]` | `None` (equal weights given) | Weight to give each representative date. If provided, must have length `num_reps` |
| `options_dict` | `dict` | `None` | Options passed to the Prescient data loader |
| `start_date` | `str` | `None` | Defines the year for the simulation (if not provided, reads from `simulation_objects.csv`) |

In [5]:
rep_days = [
    "2020-01-28 00:00",
    "2020-04-23 00:00",
    "2020-07-05 00:00",
    "2020-10-14 00:00",
]
rep_weights = {
    "2020-01-28 00:00": 115,
    "2020-04-23 00:00": 95,
    "2020-07-05 00:00": 50,
    "2020-10-14 00:00": 105,
}

data_object.load_prescient(
    data_path,
    representative_dates=rep_days,
    representative_weights=rep_weights,
)

Now that our data is loaded, it is stored under the `md` attribute of the `ExpansionPlanningData` object. By printing it, we can see it is an EGRET `ModelData` object (EGRET is a Pyomo-based tool for electrical grid optimization; we borrow its data structure here). We can also explore the contents of this `ModelData` object and inspect the data from our CSV files. 

In [6]:
print(data_object.md, "\n")

elements = data_object.md.data["elements"]
print(elements.keys(), "\n")

for gen, gen_data in elements["generator"].items():
    print(
        f"{gen}    \t{gen_data['bus']}"
        f"     \t{gen_data['generator_type']}"
        f"    \t{gen_data['unit_type']}"
    )


dict_keys(['bus', 'load', 'shunt', 'area', 'branch', 'generator', 'storage']) 

3_CT    	bus3     	thermal    	CT
10_STEAM    	bus10     	thermal    	STEAM
4_CC    	bus4     	thermal    	CC
4_STEAM    	bus4     	thermal    	STEAM
10_PV    	bus10     	renewable    	PV
2_RTPV    	bus2     	renewable    	RTPV
1_HYDRO    	bus1     	renewable    	HYDRO
4_WIND    	bus4     	renewable    	WIND
10_PV-c    	bus10     	renewable    	PV
2_RTPV-c    	bus2     	renewable    	RTPV
1_HYDRO-c    	bus1     	renewable    	HYDRO
4_WIND-c    	bus4     	renewable    	WIND



### 1b. The `DataProcessing` class

A model can be built with GTEP with only the `ExpansionPlanningData` object, but default cost values will be assumed. If you want to read in true cost data, you can leverage the `DataProcessing` class.

Its constructor takes no arguments; simply call `DataProcessing()`. To load cost data, use its `load_gen_data` method:

| Name | Type | Default value | Description |
| --- | --- | --- | --- |
| `bus_data_path` | `pathlib.Path` or `str` | - | Path to bus data |
| `cost_data_path` | `pathlib.Path` or `str` | - | Path to cost data |
| `ng_cost_path` | `pathlib.Path` or `str` | - | Path to natural gas cost data |
| `candidate_gens` | `list[str]` | - | Generator types to extract cost data for |
| `years` | `list[int]` | `[2025, 2030, 2035]` | Years to extract cost data for |
| `scenario` | `str` | `"Moderate"` | Cost scenario |
| `ng_cost_quantity` | `str` | `"Reference case"` | Natural gas cost quantity to use |
| `save_csv` | `bool` | `False` | Whether to save the resulting dataframe to csv |
| `out_path` | `pathlib.Path` or `None` | `None` | Directory to save the csv to |

After running this method, the cost data is accessible via the `gen_data_target` parameter.

Below, we show an example:

In [7]:
from gtep.gtep_data_processing import DataProcessing

bus_data_path = Path(
    "../../../../gtep/data/costs/"
    "Bus_data_gen_weights_mappings.csv"
)
cost_data_path = Path(
    "../../../../gtep/data/costs/"
    "2022_v3_Annual_Technology_Baseline_Workbook_Mid-year_update_2-15-2023_Clean.xlsx"
)
ng_cost_path = Path(
    "../../../../gtep/data/costs/"
    "Total_Energy_Supply_Disposition_and_Price_Summary.csv"
)

display_markdown_of_paths([bus_data_path, cost_data_path, ng_cost_path])

candidate_gens = [
    "Natural Gas_FE",
    "Solar - Utility PV",
    "Land-Based Wind",
]

cost_data = DataProcessing()
cost_data.load_gen_data(
    bus_data_path=bus_data_path,
    cost_data_path=cost_data_path,
    ng_cost_path=ng_cost_path,
    candidate_gens=candidate_gens,
)

cost_data.gen_data_target.iloc[:5,:10]

- [Bus_data_gen_weights_mappings.csv](../../../../gtep/data/costs/Bus_data_gen_weights_mappings.csv)
- [2022_v3_Annual_Technology_Baseline_Workbook_Mid-year_update_2-15-2023_Clean.xlsx](../../../../gtep/data/costs/2022_v3_Annual_Technology_Baseline_Workbook_Mid-year_update_2-15-2023_Clean.xlsx)
- [Total_Energy_Supply_Disposition_and_Price_Summary.csv](../../../../gtep/data/costs/Total_Energy_Supply_Disposition_and_Price_Summary.csv)

,GEN UID,Bus ID,Unit Type,Fuel,PMax MW,PMin MW,Min Up Time Hr,Min Down Time Hr,capex_2025,fixed_ops_2025
0,ct_fe1-c,1,CT,G,992,297.6,6,8,958.305641,26.87
1,ct_fe18-c,18,CT,G,992,297.6,6,8,958.305641,26.87
2,ct_fe23-c,23,CT,G,992,297.6,6,8,958.305641,26.87
3,ct_fe32-c,32,CT,G,992,297.6,6,8,958.305641,26.87
4,ct_fe36-c,36,CT,G,992,297.6,6,8,958.305641,26.87


## 2. Creating the model

The purpose of the `ExpansionPlanningModel` class is to build a Pyomo model for a grid expansion planning problem. The arguments of its constructor determine the physical setup of the problem (via the `ExpansionPlanningData` object from Step 1), as well as various cost and modeling options.

Specifically, the constructor takes the following arguments:

| Name | Type | Default value | Description |
| --- | --- | --- | --- |
| `data` | `ExpansionPlanningData` | - | Model data from Step 1a |
| `cost_data` | `DataProcessing` | `None` | Cost data from Step 1b |
| `config` | `dict` | `{}` | Model configuration options. Alternative config options will be explored later in this notebook |

In [8]:
from gtep.gtep_model import ExpansionPlanningModel

mod_object = ExpansionPlanningModel(
    data=data_object,
    cost_data=cost_data,
    config={"scale_loads": False},
)

Once the `ExpansionPlanningModel` object is created, we call its `create_model` method to build the corresponding Pyomo model.

In [9]:
mod_object.create_model()

[    0.00] Creating GTEP Model


When `create_model` is called, several things happen behind the scenes.

Initial setup:
- A Pyomo `ConcreteModel` object is created and saved to `mod_object.model`
- The cost object is saved to `mod_object.model.mc`
- The `ExpansionPlanningData` object is saved to `mod_object.model.data`

Then, the model is constructed as follows:
1. __Sets are added to the Pyomo model__, used to index other Pyomo objects like parameters, blocks, variables, and constraints. This includes sets for grid components (`lines`, `thermalGenerators`, etc.) as well as time periods (`stages`, `representativePeriods`, etc.).
2. __Parameters are added to the Pyomo model__, primarily quantities from the `ExpansionPlanningData` object. For instance, `from_bus` and `to_bus` identify which buses each line connects to, and `thermalCapacity` stores the "p_max" value for each thermal generator.
3. __Stages are added in a nested manner__, reflecting the temporal structure of GTEP (adding the relevant parameters, variables, constraints, and disjuncts for each):
    1. The investment stages are added to the Pyomo model.
    2. The representative periods are added to each investment stage.
    3. The commitment periods are added to each representative period.
    4. The dispatch periods are added to each commitment period.
4. Finally, __the objective is added to the Pyomo model.__

Before moving on, we can see that indeed the Pyomo model, the `ExpansionPlanningData` object, and the EGRET data object are all present:

In [10]:
print(type(mod_object.model))
print(type(mod_object.model.data))
print(type(mod_object.model.md))

<class 'pyomo.core.base.PyomoModel.ConcreteModel'>
<class 'gtep.gtep_data.ExpansionPlanningData'>
<class 'egret.data.model_data.ModelData'>


## 3. Solving the model

Once the Pyomo model has been created, it is up to the user to apply any required model transformations and send the model to a solver. At a minimum, to solve a GDP-based model like the one produced by GTEP, we need to do two things:

1. Transform the model into a solvable form (we will use `"gdp.bigm"`)
2. Pass the transformed model to a solver (we will use `"highs"`)

Note that Pyomo is compatible with several solvers, including some that require a license (e.g., Gurobi). Solvers aren't included in Pyomo or GTEP by default and must be installed into your Python environment manually. For the sake of this tutorial, we have already added HiGHS, a freely available solver compatible with Pyomo, to your environment (Python interface: `highspy`).

For more information on solving GDP models, refer to https://pyomo.readthedocs.io/en/stable/explanation/modeling/gdp/solving.html.

The object returned by `opt.solve` stores useful information, including the termination condition. This can be accessed programmatically to check that a valid solution was found:

In [11]:
from contextlib import redirect_stdout, redirect_stderr

with open("output.log", "w") as f, redirect_stdout(f), redirect_stderr(f):
    from pyomo.environ import SolverFactory, TransformationFactory

    TransformationFactory("gdp.bigm").apply_to(mod_object.model)  # apply transformation here
    opt = SolverFactory("highs")  # select your solver here
    result = opt.solve(mod_object.model, tee=True)  # tee=True prints solver updates

In [12]:
for solver_result in result["Solver"]:
    print(solver_result)


Status: ok
Termination condition: optimal
Termination message: TerminationCondition.convergenceCriteriaSatisfied



## 4. Exploring results

One way to probe the solution is to manually investigate model components. For instance, `total_cost_objective` is the objective in GTEP. `pyomo.environ` provides the `value` function to extract the value of a component:

In [13]:
from pyomo.environ import value
value(mod_object.model.total_cost_objective)

3168393.4448161027

Alternatively, calling `display()` on a Pyomo component produces a nicely formatted output:

In [14]:
mod_object.model.total_cost_objective.display()

total_cost_objective : Size=1, Index=None, Active=True
    Key  : Active : Value
    None :   True : 3168393.4448161027


It's important to note that printing the component itself (or letting Jupyter display it) won't give you the value:

In [15]:
print(mod_object.model.total_cost_objective)
mod_object.model.total_cost_objective

total_cost_objective


`value` does not work on indexed components. Instead, we must first access the component at a valid index element. See below:

In [16]:
def get_value(x):
    try:
        print("My value:", value(x))
        print(f"I am a {type(x)} -- passing me to value() works")
    except TypeError:
        print(f"I am a {type(x)} -- passing me to value() does not work")

print("-" * 50, "\nEXAMPLE 1 (passing whole component)")
component = mod_object.model.investmentStage[1].renewableOperational
get_value(component)  # doesn't work

print("\n" + "-" * 50, "\nEXAMPLE 2 (passing one element of component)")

for i in component:  # this loops through the index set of component
    get_value(component[i])
    break

print("\n" + "-" * 50, "\nEXAMPLE 3 (calling the display method)")
component.display()
print(
    "This works to print the values, but can't access them "
    "programmatically (doesn't return anything)"
)

print("\n" + "-" * 50, "\nEXAMPLE 4 (calling the extract_values method)")
vals_dict = component.extract_values()
print(vals_dict)
print("This returns a dictionary mapping index elements to values")

-------------------------------------------------- 
EXAMPLE 1 (passing whole component)
ERROR: evaluating object as numeric value:
investmentStage[1].renewableOperational
        (object: <class 'pyomo.core.base.var.IndexedVar'>)
    'IndexedVar' object is not callable
I am a <class 'pyomo.core.base.var.IndexedVar'> -- passing me to value() does not work

-------------------------------------------------- 
EXAMPLE 2 (passing one element of component)
My value: 20.8
I am a <class 'pyomo.core.base.var.VarData'> -- passing me to value() works

-------------------------------------------------- 
EXAMPLE 3 (calling the display method)
renewableOperational : Size=8, Index=renewableGenerators, Units=MW
    Key       : Lower : Value   : Upper : Fixed : Stale : Domain
        10_PV :     0 :    20.8 :  None :  True : False : NonNegativeReals
      10_PV-c :     0 :    20.8 :  None : False : False : NonNegativeReals
      1_HYDRO :     0 :  48.483 :  None :  True : False : NonNegativeReals
    1

Now, let's try exploring the solution manually by looking at the solved values for model variables.

For instance, we can investigate investment decisions, like which generators are operational at each investment stage `i`, by looking at:
- For thermal generators: `investmentStage[i].genOperational[thermal_generator].indicator_var` (the entire generator is either operational or not)
- For renewable generators: `investmentStage[i].renewableOperational[renewable_generator]` (a numerical operational capacity is solved for)

In [17]:
for i in mod_object.model.stages:
    print("-" * 50)
    print(f"INVESTMENT STAGE {i}")

    print("Which thermal generators are operational or extended:")
    for thermal_generator in mod_object.model.thermalGenerators:
        inv = mod_object.model.investmentStage[i]
        print(
            thermal_generator,
            "   \t",
            (
                value(inv.genOperational[thermal_generator].indicator_var)
                or value(inv.genExtended[thermal_generator].indicator_var)
            ),
        )

    print("Renewables operational plus extended generation capacity:")
    for renewable_generator in mod_object.model.renewableGenerators:
        print(
            renewable_generator,
            "   \t",
            (
                value(inv.renewableOperational[renewable_generator])
                + value(inv.renewableExtended[renewable_generator])
            ),
        )

--------------------------------------------------
INVESTMENT STAGE 1
Which thermal generators are operational or extended:
3_CT    	 True
10_STEAM    	 True
4_CC    	 True
4_STEAM    	 True
Renewables operational plus extended generation capacity:
10_PV    	 20.8
2_RTPV    	 7.8
1_HYDRO    	 48.483
4_WIND    	 118.486
10_PV-c    	 20.8
2_RTPV-c    	 0.0
1_HYDRO-c    	 48.483
4_WIND-c    	 118.486


We can also look at dispatch-level variables (for instance, the amount of power generated by each generator) by accessing the `thermalGeneration` and `renewableGeneration` variables on each dispatch block.

Below, we pull out a dispatch block from each representative period and print the generation values.

In [18]:
dispatch_blocks = {
    r: mod_object.model
        .investmentStage[1]
        .representativePeriod[r]
        .commitmentPeriod[1]
        .dispatchPeriod[1]
    for r in mod_object.model.representativePeriods
}

for r, b in dispatch_blocks.items():
    print("-" * 50)
    print(f"REPRESENTATIVE PERIOD {r}")
    print("Generator\t Generation (MW)")
    
    for generator, power in b.thermalGeneration.items():
        print(generator, "   \t", value(power))
    for generator, generation in b.renewableGeneration.items():
        print(generator, "   \t", value(generation))

--------------------------------------------------
REPRESENTATIVE PERIOD 1
Generator	 Generation (MW)
3_CT    	 0.0
10_STEAM    	 0.0
4_CC    	 0.0
4_STEAM    	 0.0
10_PV    	 0.0
2_RTPV    	 0.0
1_HYDRO    	 13.658
4_WIND    	 0.0
10_PV-c    	 0.0
2_RTPV-c    	 0.0
1_HYDRO-c    	 13.658
4_WIND-c    	 42.87099999999998
--------------------------------------------------
REPRESENTATIVE PERIOD 2
Generator	 Generation (MW)
3_CT    	 0.0
10_STEAM    	 0.0
4_CC    	 42.447000000000045
4_STEAM    	 0.0
10_PV    	 0.0
2_RTPV    	 0.0
1_HYDRO    	 10.591999999999997
4_WIND    	 2.186
10_PV-c    	 0.0
2_RTPV-c    	 0.0
1_HYDRO-c    	 10.592
4_WIND-c    	 2.186
--------------------------------------------------
REPRESENTATIVE PERIOD 3
Generator	 Generation (MW)
3_CT    	 14.164999999999878
10_STEAM    	 76.00000000000011
4_CC    	 0.0
4_STEAM    	 0.0
10_PV    	 0.0
2_RTPV    	 0.0
1_HYDRO    	 12.3
4_WIND    	 0.639
10_PV-c    	 0.0
2_RTPV-c    	 0.0
1_HYDRO-c    	 12.3
4_WIND-c    	 0.639
-----

To streamline extracting this data, we can leverage the `ExpansionPlanningSolution` class. Its constructor takes a single required argument (the path to the data we used to construct the model).

| Name | Type | Default value | Description |
| --- | --- | --- | --- |
| `data_path` | `Path` or `str` | - | Path to model data from Step 1 |

In [19]:
from gtep.gtep_solution import ExpansionPlanningSolution

soln = ExpansionPlanningSolution(data_path)

We can then call the class's `save_results_in_json_files` method to automatically write out data from the model solution. It takes three arguments:

| Name | Type | Default value | Description |
| --- | --- | --- | --- |
| `gtep_model` | `ExpansionPlanningModel` | - | Model object we solved in Step 3 |
| `dir_name` | `Path` or `str` | - | Path to write solution files to |
| `value_threshold` | `float` | 0.001 | Threshold below which values are not written |

In [20]:
soln_path = Path("soln")
soln.save_results_in_json_files(mod_object, soln_path)

The `ExpansionPlanningSolution` class has a `create_plots` method, which produces interactive html plots to help understand decisions made by the model. It takes four arguments:

| Name | Type | Default value | Description |
| --- | --- | --- | --- |
| `case_json` | `str` | - | Which data to read from json; should be one of `"renewables"`, `"dispatchables"`, or `"combined"` (which reads both) |
| `results_path` | `Path` or `str` | - | Path to results we just saved |
| `data_path` | `Path` or `str` | - | Path to model data from Step 1 |
| `plot_type` | `str` | `"all"` | Type of plot to make; must be one of: `"treemap"`, `"piechart"`, or `"all"` (which makes both) |

In [21]:
pie = soln.create_plots("combined", soln_path, data_path, "piechart", savefig=False)

Before displaying the figure, let's define a helper function to convert the figure to HTML. This makes the plot display more consistently across platforms.

In [22]:
from IPython.display import display, HTML
from copy import copy

def display_plotly_as_html(fig, title=None):
    if title is not None:
        fig = copy(fig)
        fig.update_layout(title=title)
    fig_html = fig.to_html(full_html=False, include_plotlyjs="cdn")
    return display(HTML(fig_html))

display_plotly_as_html(pie)

The `ExpansionPlanningSolution` class also has a `create_stackgraph` method, which produces another plot based on the representative days.
| Name | Type | Default value | Description |
| --- | --- | --- | --- |
| `results_path` | `Path` or `str` | - | Path to results we just saved |
| `rep_days` | `list[str]` | - | Representative days |

In [23]:
stackgraph = soln.create_stackgraph(soln_path, rep_days, savefig=False)
display_plotly_as_html(stackgraph)

In the first representative period (the first two columns), wind is able to meet load. Since it does not have associated fuel costs, it is selected over generation types (e.g., gas). In the second representative period (the second two columns), hydro and stream are also used to meet the load.

## Exploring GTEP modeling options

### Experiment 1

In this experiment, we will set `include_investment=False`.

Note that while investment blocks are included in the model structure regardless of the value of `include_investment`, when `include_investment=False`, candidate elements are not added.

First, let's define a helper function to solve the model and write out solution details:

In [24]:
def solve_model_and_make_plots(model_object, data_path, write_dir):
    
    # transform and solve
    TransformationFactory("gdp.bigm").apply_to(model_object.model)
    result_object = opt.solve(model_object.model)

    # check termination condition
    term_cond = result_object["Solver"][0]["Termination condition"]
    print("Termination condition:", term_cond)
    if term_cond != "optimal":
        return

    # make solution
    soln_object = ExpansionPlanningSolution(data_path)
    soln_object_path = (Path() / write_dir).resolve()
    soln_object.save_results_in_json_files(model_object, soln_object_path)
    pie = soln_object.create_plots(
        "combined", soln_object_path, data_path, "piechart", savefig=False
    )
    rep_days = [
        value(model_object.model.representativeDate[idx])
        for idx in model_object.model.representativeDate
    ]
    stackgraph = soln_object.create_stackgraph(
        soln_object_path, rep_days, savefig=False
    )
    return pie, stackgraph

Now, we can solve our model with no investment and compare to the baseline case:

In [25]:
mod_no_invest = ExpansionPlanningModel(
    data=data_object,
    cost_data=cost_data,
    config={"scale_loads": False, "include_investment": False},
)
mod_no_invest.create_model()

pie_no_invest, stackgraph_no_invest = solve_model_and_make_plots(
    mod_no_invest,
    data_path,
    "soln_no_invest"
)

print(
    "Objective with investment (baseline):",
    value(mod_object.model.total_cost_objective)
)
print(
    "Objective without investment (experiment 1):",
    value(mod_no_invest.model.total_cost_objective)
)

display_plotly_as_html(pie, title="Baseline")
display_plotly_as_html(pie_no_invest, title="Experiment 1 (no investment)")

display_plotly_as_html(stackgraph, title="Baseline")
display_plotly_as_html(stackgraph_no_invest, title="Experiment 1 (no investment)")

[    0.00] Creating GTEP Model
Termination condition: optimal
Objective with investment (baseline): 3168393.4448128548
Objective without investment (experiment 1): 3767479.0251299744


Observations:
- Without investment, Wind and Hydro make up a significantly smaller portion of the generation mix compared to the baseline case (see the pie charts). The other generation types instead account for a larger portion of the generation mix.
- In the stackgraph, while Wind and Hydro still contribute in the experiment, they cannot contribute the same amount of power as they did in the baseline case without candidates. Instead, Steam largely makes up the difference.

### Experiment 2

In this experiment, we will change the number of representative periods. Note we'll have to create a new `ExpansionPlanningData` object for this change.

In [26]:
data_object_fewer_repr_days = ExpansionPlanningData(
    stages=1,
    num_reps=2,  # was 4
    num_commit=6,
    num_dispatch=4,
    duration_representative_period=6,
)

data_object_fewer_repr_days.load_prescient(
    data_path,
    representative_dates=[
        "2020-01-28 00:00",
        "2020-07-05 00:00",
    ],
    representative_weights={
        "2020-01-28 00:00": 210,
        "2020-07-05 00:00": 155,
    },
)

mod_fewer_repr_days = ExpansionPlanningModel(
    data=data_object_fewer_repr_days,
    cost_data=cost_data,  # use the same cost data object
    config={"scale_loads": False},
)
mod_fewer_repr_days.create_model()

pie_fewer_repr_days, stackgraph_fewer_repr_days = solve_model_and_make_plots(
    mod_fewer_repr_days,
    data_path,
    "soln_fewer_repr_days",
)

print(
    "Objective with 4 representative dates:",
    value(mod_object.model.total_cost_objective)
)
print(
    "Objective with 2 representative dates:",
    value(mod_fewer_repr_days.model.total_cost_objective)
)

display_plotly_as_html(pie, title="Baseline")
display_plotly_as_html(pie_fewer_repr_days, title="Experiment 2 (fewer representative days)")

display_plotly_as_html(stackgraph, title="Baseline")
display_plotly_as_html(stackgraph_fewer_repr_days, title="Experiment 2 (fewer representative days)")

[    0.00] Creating GTEP Model
Termination condition: optimal
Objective with 4 representative dates: 3168393.4448128548
Objective with 2 representative dates: 3265990.3817659924


We can make a few observations:
- In the pie charts, this experiment shows less Solar and especially Hydro capacity than the baseline case. The other generation types make up a larger portion of the overall generation mix.
- In the stackgraphs, the generation profiles looks roughly similar for both the January 28th representative period and the July 5th representative period. Note that the April 23rd representative period used the most Hydro in the baseline case; without that representative period in the experiment, we chose to invest less in Hydro.